# Ingest races.csv file
### 1. Read the file using spark dataframe reader API
### 2. Add Metadata Columns 
-       Source File
-       Ingestion Timestamp
### 3. Write to bronze delta table

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
%run  ../00-common/01_Environmnet_config

In [0]:
%run  ../00-common/02_bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    DateType,
)

races_schema = StructType(
    [
        StructField("season", IntegerType()),
        StructField("round", IntegerType()),
        StructField("url", StringType()),
        StructField("raceName", StringType()),
        StructField("date", DateType()),
        StructField("circuitId", StringType()),
    ]
)

In [0]:
races_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(races_schema)
    .option("mode", "FAILFAST")
    .load(source_file)
)

In [0]:
races_final_df = add_ingestion_medatat(races_df)

In [0]:
(races_final_df.write.format("delta").mode("overwrite").saveAsTable(table_name))

In [0]:
spark.sql(f"select * from {table_name}").display()